# PRB Drift Diagnosis — Why start ≠ end, and what happens after handover

## Questions answered

1. **Why do used PRBs at the start of an episode differ from the end — even with fixed demand profiles?**
2. **Is the growth in the logs a real state change or a logging artifact?**
3. **After a handover across different slices (eMBB / URLLC / mMTC), do total PRBs increase or decrease?**

---

## Root causes (read before running)

### Why PRBs drift inside one episode

There are **two distinct PRB quantities** in play:

| Variable | What it is | Fixed? |
|---|---|---|
| `ue.upper_demand_prbs` | Requested PRBs / tick (set at scenario init) | **Yes — conserved across the episode** |
| `ue.useful_prbs` | PRBs actually used by the scheduler last substep | **No — updated every substep** |
| `window_average_useful_prbs` | Mean `useful_prbs` over the measurement window | **No — changes with queue state** |

The reward and logs use **window_average_useful_prbs**, which is NOT the same as demand PRBs.

### The calibration feedback loop (`_calibrate_demand_from_radio_window`)

After every upper window, `step()` calls `_calibrate_demand_from_radio_window()`.  
This updates **`source.bit_rate`** (the traffic generator) using:

```
correction = clip(desired_prbs / achieved_prbs,  0.25, 4.0)
new_rate   = old_rate × [(1-α) + α × correction]      (α = 0.5 by default)
```

Consequences:
- **Congested cell**: scheduler delivers fewer PRBs than demanded → correction > 1 → rate rises → queue grows → more useful_prbs in next window
- **Light cell**: scheduler over-delivers → correction < 1 → rate falls → queue shrinks → fewer useful_prbs
- This is a real state change (not a logging artifact).  The logs correctly reflect what is happening.
- With α=0.5 the loop converges in 3–5 windows.  Before convergence, start ≠ end.

### Why handover changes PRBs (even with the same demand)

When a UE moves from gNB-A to gNB-B:
1. `upper_demand_prbs` follows the UE (conserved)
2. SINR at gNB-B is different → `bits_per_prb` is different
3. **`_recalibrate_handover_ues()` is never called** (it is defined but has no caller).
   The old `source.bit_rate` — calibrated for gNB-A — remains for the first post-HO window.
4. If gNB-B has **better SINR**: same bit_rate now needs fewer PRBs → total useful PRBs **decrease** transiently
5. If gNB-B has **worse SINR**: same bit_rate needs more PRBs → total **increases** transiently
6. `_calibrate_demand_from_radio_window` corrects the bit rate over the next 2–3 windows back toward `upper_demand_prbs`


In [ ]:
import os, sys
from pathlib import Path

os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib-chech')
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from scenario_creator import create_multignb_env
from global_ppo_3gnb_env import GlobalPPO3GNBEnv
from local_a3_agent_wrapper import normalize_slice_type

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
SLICE_TYPES = ('eMBB', 'URLLC', 'mMTC')
print('Ready')

---
## Part 1 — PRB drift with ZERO handovers (fixed demand)

Run the upper env for 10 steps with a zero action (no offsets applied).  
Since no handovers fire, `upper_demand_prbs` never changes.  
We track `demand_prbs` (fixed) vs `useful_prbs` (window average) per step.

In [ ]:
N_STEPS = 15

env_drift = GlobalPPO3GNBEnv(
    seed=42,
    scenario_mode='snapshot',
    snapshot_scenario='embb_g0_offload',   # gNB-0 heavily loaded
    local_steps_per_global=10,
    global_steps_per_episode=N_STEPS,
    radio_substeps=50,
    upper_window_seconds=1.0,
    warmup_steps=2,
    post_handover_settle_steps=4,
    demand_calibration_alpha=0.5,
    safe_admission_enabled=False,
    terminal_reward_only=False,
)

obs, info = env_drift.reset()
print(f'Scenario: {info["scenario_name"]}  |  episode steps: {env_drift._active_episode_steps}')

# Show initial UE demand PRBs
all_ues = env_drift.base_env.get_all_ues()
print(f'\n{len(all_ues)} UEs after reset:')
total_demand = 0
for ue in all_ues:
    d = int(getattr(ue, 'upper_demand_prbs', 0))
    total_demand += d
    print(f'  UE-{ue.id}  slice={ue.slice_type}  gNB={ue.serving_gnb}  '
          f'demand_prbs={d}  bit_rate={getattr(getattr(ue,"traffic_source",None),"bit_rate",0)/1e6:.2f} Mbps')
print(f'  Total demand PRBs = {total_demand}')

In [ ]:
zero_action = np.zeros(env_drift.action_space.shape, dtype=np.float32)
drift_records = []

for step in range(N_STEPS):
    obs, reward, terminated, truncated, info = env_drift.step(zero_action)

    # demand PRBs (conserved, fixed per UE)
    demand_per_gnb = info['jain_per_gnb_demand_prbs']   # shape (3,)
    total_demand_prbs = int(demand_per_gnb.sum())

    # useful PRBs from window average  (what the reward uses)
    used_prb_end = info['used_prb_matrix_end']           # shape (3,3)
    total_useful_prbs = float(used_prb_end.sum())        # sum over gnbs × slices (already in PRB units)

    # per-gNB total load (useful, normalized)
    gnb_useful_prbs = np.sum(used_prb_end, axis=1)       # shape (3,)

    # current bit rates from traffic generators
    bit_rates = [
        getattr(getattr(ue, 'traffic_source', None), 'bit_rate', 0.0)
        for ue in env_drift.base_env.get_all_ues()
        if ue.connected
    ]

    # handover check  (should be 0 with zero action)
    n_handovers = int(info['handover_count'])

    drift_records.append({
        'step': step + 1,
        'total_demand_prbs': total_demand_prbs,
        'total_useful_prbs': total_useful_prbs,
        'gnb0_useful': float(gnb_useful_prbs[0]),
        'gnb1_useful': float(gnb_useful_prbs[1]),
        'gnb2_useful': float(gnb_useful_prbs[2]),
        'mean_bit_rate_mbps': float(np.mean(bit_rates)) / 1e6 if bit_rates else 0.0,
        'n_handovers': n_handovers,
        'jain_raw': float(info['jain_fairness_raw']),
        'reward': float(reward),
    })
    print(f'step={step+1:2d}  demand_prbs={total_demand_prbs:4d}  '
          f'useful_prbs={total_useful_prbs:6.1f}  '
          f'mean_rate={drift_records[-1]["mean_bit_rate_mbps"]:.3f} Mbps  '
          f'HOs={n_handovers}')

    if truncated:
        break

env_drift.close()
df_drift = pd.DataFrame(drift_records)
print(f'\nDemand PRBs range: {df_drift.total_demand_prbs.min()} – {df_drift.total_demand_prbs.max()}  (should be constant)')
print(f'Useful PRBs range: {df_drift.total_useful_prbs.min():.1f} – {df_drift.total_useful_prbs.max():.1f}  (will drift!)')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

ax_prb, ax_rate, ax_gap = axes

ax_prb.axhline(df_drift.total_demand_prbs.iloc[0], color='black', lw=1.5,
               linestyle='--', label=f'demand_prbs (fixed = {df_drift.total_demand_prbs.iloc[0]})')
ax_prb.plot(df_drift.step, df_drift.total_useful_prbs, 'o-', color='steelblue',
            lw=2, label='useful_prbs (window avg)')
ax_prb.fill_between(df_drift.step, df_drift.total_useful_prbs,
                    df_drift.total_demand_prbs.iloc[0], alpha=0.12, color='steelblue')
ax_prb.set_ylabel('Total PRBs (network)')
ax_prb.legend()
ax_prb.grid(alpha=0.4)
ax_prb.set_title('PRB drift during episode — ZERO action (no handovers), fixed demand profile')

ax_rate.plot(df_drift.step, df_drift.mean_bit_rate_mbps, 's-', color='tomato',
             lw=2, label='mean UE bit_rate (Mbps)')
ax_rate.set_ylabel('Mean UE traffic rate (Mbps)')
ax_rate.legend()
ax_rate.grid(alpha=0.4)
ax_rate.set_title('_calibrate_demand_from_radio_window() adjusts source.bit_rate each step')

gap = df_drift.total_demand_prbs - df_drift.total_useful_prbs
ax_gap.bar(df_drift.step, gap, color=np.where(gap > 0, 'tomato', 'seagreen'), alpha=0.7)
ax_gap.axhline(0, color='black', lw=0.8)
ax_gap.set_ylabel('demand − useful PRBs')
ax_gap.set_xlabel('Upper step')
ax_gap.set_title('Positive = scheduler under-delivers → calibration increases rate next step')
ax_gap.grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.savefig('prb_drift_no_handover.png', dpi=130, bbox_inches='tight')
plt.show()
print('Key takeaway: demand_prbs is flat; useful_prbs changes because the calibration')
print('feedback loop adjusts source.bit_rate at every step until it converges.')

---
## Part 2 — PRB change after handover across different slices

Setup: 3 gNBs, 3 slices (eMBB / URLLC / mMTC).  
Initial state: gNB-0 heavily loaded for each slice.  
At step 4 we apply a strong negative bias on gNB-0 → gNB-1 direction for all slices,  
triggering real A3 handovers.  

**What to watch:**
- Per-gNB demand PRBs (conserved, just re-assigned)
- Per-gNB useful PRBs (may spike or dip post-HO due to SINR change + calibration lag)
- Total network PRBs — should remain close to the demand total in the long run

In [ ]:
# 12 steps total: 3 steps baseline, 1 step with strong HO bias, 8 steps recovery
N_STEPS_HO = 12
HO_AT_STEP = 3    # 0-indexed: apply strong offload at step index 3

env_ho = GlobalPPO3GNBEnv(
    seed=7,
    scenario_mode='snapshot',
    # Scenario: gNB-0 overloaded for eMBB, gNB-1 overloaded for URLLC, gNB-2 overloaded for mMTC
    snapshot_scenario='multi_slice_multi_gnb_congestion',
    local_steps_per_global=10,
    global_steps_per_episode=N_STEPS_HO,
    radio_substeps=50,
    upper_window_seconds=1.0,
    warmup_steps=2,
    post_handover_settle_steps=4,
    demand_calibration_alpha=0.5,
    safe_admission_enabled=True,
    safe_admission_load_limits={'eMBB': 0.80, 'URLLC': 0.80, 'mMTC': 0.80},
    terminal_reward_only=False,
    max_handovers_per_episode=30,
    max_handovers_per_ue_episode=2,
)

obs, info = env_ho.reset()
print(f'Scenario: {info["scenario_name"]}')

ues = env_ho.base_env.get_all_ues()
print(f'{len(ues)} UEs after reset:')
for ue in ues:
    d = int(getattr(ue, 'upper_demand_prbs', 0))
    print(f'  UE-{ue.id}  slice={ue.slice_type}  gNB={ue.serving_gnb}  demand_prbs={d}')

In [ ]:
# Strong offload action: gNB-0 releases all slices toward gNB-1
# Action tensor shape: (n_gnbs=3, max_neighbors=2, n_slices=3)
# Flattened: gNB-0→neighbor0, gNB-0→neighbor1, gNB-1→..., gNB-2→...
# neighbors: {0: [1,2], 1: [0,2], 2: [0,1]}
# negative bias = push UEs away (release toward neighbor)

zero_action = np.zeros(env_ho.action_space.shape, dtype=np.float32)

# Strong offload: -0.9 for gNB-0 in direction of each neighbor, all slices
ho_action = np.zeros((3, 2, 3), dtype=np.float32)
ho_action[0, :, :] = -0.9    # gNB-0: push all slices toward both neighbors
ho_action = ho_action.reshape(-1)

ho_records = []

for step in range(N_STEPS_HO):
    action = ho_action if step == HO_AT_STEP else zero_action
    obs, reward, terminated, truncated, info = env_ho.step(action)

    # demand PRBs per gNB (conserved, follows UE)
    demand_per_gnb = info['jain_per_gnb_demand_prbs']

    # useful PRBs (window average) per gNB × slice  [shape (3,3)]
    used_prb_end = info['used_prb_matrix_end']
    gnb_useful = np.sum(used_prb_end, axis=1)

    # useful PRBs per slice (across all gNBs)
    slice_useful = np.sum(used_prb_end, axis=0)

    # demand PRBs per slice (sum demand_prbs of UEs per slice)
    slice_demand = {s: 0 for s in SLICE_TYPES}
    for ue in env_ho.base_env.get_all_ues():
        if ue.connected and ue.serving_gnb is not None:
            st = normalize_slice_type(getattr(ue, 'slice_type', 'eMBB'))
            slice_demand[st] += int(getattr(ue, 'upper_demand_prbs', 0))

    ho_records.append({
        'step': step + 1,
        'is_ho_step': (step == HO_AT_STEP),
        # demand per gNB
        'demand_gnb0': int(demand_per_gnb[0]),
        'demand_gnb1': int(demand_per_gnb[1]),
        'demand_gnb2': int(demand_per_gnb[2]),
        'total_demand': int(demand_per_gnb.sum()),
        # useful per gNB
        'useful_gnb0': float(gnb_useful[0]),
        'useful_gnb1': float(gnb_useful[1]),
        'useful_gnb2': float(gnb_useful[2]),
        'total_useful': float(gnb_useful.sum()),
        # per-slice useful
        'useful_embb':  float(slice_useful[0]),
        'useful_urllc': float(slice_useful[1]),
        'useful_mmtc':  float(slice_useful[2]),
        # per-slice demand
        'demand_embb':  slice_demand['eMBB'],
        'demand_urllc': slice_demand['URLLC'],
        'demand_mmtc':  slice_demand['mMTC'],
        # handovers this step
        'n_handovers': int(info['handover_count']),
        'jain': float(info['jain_fairness_raw']),
        'reward': float(reward),
    })

    flag = ' <-- HO ACTION' if step == HO_AT_STEP else ''
    print(f'step={step+1:2d}  '
          f'demand=[{demand_per_gnb[0]:3d},{demand_per_gnb[1]:3d},{demand_per_gnb[2]:3d}]  '
          f'useful=[{gnb_useful[0]:5.1f},{gnb_useful[1]:5.1f},{gnb_useful[2]:5.1f}]  '
          f'HOs={info["handover_count"]:2d}  '
          f'jain={info["jain_fairness_raw"]:.3f}{flag}')

    if truncated:
        break

env_ho.close()
df_ho = pd.DataFrame(ho_records)

In [ ]:
# ── Figure 1: demand vs useful PRBs per gNB ──────────────────────────────────
gnb_colors = ['steelblue', 'tomato', 'seagreen']
steps = df_ho.step.values
ho_line = HO_AT_STEP + 1   # 1-indexed step number where HO fires

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

ax_demand, ax_useful = axes

for g, color in enumerate(gnb_colors):
    ax_demand.step(steps, df_ho[f'demand_gnb{g}'], where='post',
                   color=color, lw=2.5, label=f'gNB-{g} demand')
    ax_useful.plot(steps, df_ho[f'useful_gnb{g}'], 'o-',
                   color=color, lw=2.0, ms=5, label=f'gNB-{g} useful')

for ax in axes:
    ax.axvline(ho_line, color='crimson', lw=2, linestyle='--', label='HO action')
    ax.axvline(ho_line + 1, color='crimson', lw=1, linestyle=':', alpha=0.5, label='1st post-HO step')
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(alpha=0.4)

ax_demand.set_ylabel('Demand PRBs per gNB')
ax_demand.set_title('Demand PRBs (conserved, follows UE after handover)')

ax_useful.set_ylabel('Useful PRBs per gNB\n(window average)')
ax_useful.set_xlabel('Upper step')
ax_useful.set_title('Useful PRBs — immediate post-HO transient, then convergence')

plt.suptitle('Per-gNB PRBs: demand conserved but useful PRBs spike/dip after handover', y=1.01)
plt.tight_layout()
plt.savefig('prb_handover_per_gnb.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# ── Figure 2: per-slice useful vs demand ─────────────────────────────────────
slice_cols  = [('embb', 'eMBB'),  ('urllc', 'URLLC'), ('mmtc', 'mMTC')]
slice_colors = ['royalblue', 'darkorange', 'mediumseagreen']

fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)

for ax, (key, label), color in zip(axes, slice_cols, slice_colors):
    demand_col = f'demand_{key}'
    useful_col = f'useful_{key}'
    ax.axhline(df_ho[demand_col].iloc[0], color=color, lw=1.5, linestyle='--',
               label=f'demand_prbs (fixed={df_ho[demand_col].iloc[0]})')
    ax.plot(steps, df_ho[useful_col], 'o-', color=color, lw=2.0, ms=5,
            label='useful_prbs (window avg)')
    ax.fill_between(steps, df_ho[useful_col], df_ho[demand_col].iloc[0],
                    alpha=0.10, color=color)
    ax.axvline(ho_line, color='crimson', lw=2, linestyle='--', label='HO action')
    ax.set_ylabel(f'{label}\nPRBs (network)')
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(alpha=0.4)

axes[-1].set_xlabel('Upper step')
plt.suptitle('Per-slice useful PRBs across gNBs:\n'
             'demand is fixed but useful PRBs show post-HO transient before converging', y=1.01)
plt.tight_layout()
plt.savefig('prb_handover_per_slice.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# ── Figure 3: total network PRBs — conservation check ─────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

ax_total, ax_diff = axes

ax_total.axhline(df_ho.total_demand.iloc[0], color='black', lw=1.5, linestyle='--',
                 label=f'total demand_prbs (fixed={df_ho.total_demand.iloc[0]})')
ax_total.plot(steps, df_ho.total_useful, 'o-', color='purple', lw=2.5, ms=6,
              label='total useful_prbs')
ax_total.axvline(ho_line, color='crimson', lw=2, linestyle='--', label='HO action')
ax_total.set_ylabel('Total network PRBs')
ax_total.legend()
ax_total.grid(alpha=0.4)
ax_total.set_title('Total network PRBs: demand is conserved; useful PRBs may temporarily deviate')

diff = df_ho.total_demand - df_ho.total_useful
ax_diff.bar(steps, diff, color=np.where(diff.values > 0, 'tomato', 'seagreen'), alpha=0.7)
ax_diff.axhline(0, color='black', lw=0.8)
ax_diff.axvline(ho_line, color='crimson', lw=2, linestyle='--')
ax_diff.set_ylabel('demand − useful PRBs')
ax_diff.set_xlabel('Upper step')
ax_diff.set_title('Red = scheduler under-delivers (rate will increase next step);  '
                  'Green = over-delivers (rate will decrease)')
ax_diff.grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.savefig('prb_conservation_check.png', dpi=130, bbox_inches='tight')
plt.show()

print(f'\nTotal demand PRBs across ALL steps (should be constant): '
      f'{df_ho.total_demand.unique().tolist()}')
print(f'Total useful PRBs: min={df_ho.total_useful.min():.1f}  '
      f'max={df_ho.total_useful.max():.1f}')
print(f'Max deviation from demand: {abs(diff).max():.1f} PRBs '
      f'({100*abs(diff).max()/df_ho.total_demand.iloc[0]:.1f}%)')

In [ ]:
# Pretty summary table
display_cols = [
    'step', 'n_handovers',
    'demand_gnb0', 'demand_gnb1', 'demand_gnb2', 'total_demand',
    'useful_gnb0', 'useful_gnb1', 'useful_gnb2', 'total_useful',
    'jain',
]

tbl = df_ho[display_cols].copy()
for col in ['useful_gnb0', 'useful_gnb1', 'useful_gnb2', 'total_useful']:
    tbl[col] = tbl[col].round(1)
tbl['jain'] = tbl['jain'].round(3)

# mark the HO step
tbl.insert(0, 'event', '')
tbl.loc[tbl.step == ho_line, 'event'] = '*** HO ***'

print(tbl.to_string(index=False))

---
## Summary and Conclusions

### Q1 — Why are PRBs different at episode start vs end?

**`upper_demand_prbs` never changes** — it is a conserved quantity per UE.  
What changes is `useful_prbs` = actual scheduler output, which the **calibration loop adjusts every step** via `_calibrate_demand_from_radio_window()`.

The feedback:
1. Scheduler delivers fewer useful_prbs than demanded → `source.bit_rate` increases → more queue → more useful_prbs next step
2. Scheduler over-delivers → `source.bit_rate` decreases

With `α=0.5` this loop converges in 3–5 windows, which is why early episode steps look different from later ones.

### Q2 — Is the growth in the logs a real state change or a logging artifact?

**Real state change.** The logs are correct.  
The calibration actually modifies `source.bit_rate`, which causes more/less traffic to be generated,  
which changes the queue, which changes how many PRBs the scheduler allocates.  
This is NOT a logging artifact — it is intended behavior (demand tracking), but it introduces  
episode non-stationarity: the effective network load is NOT constant from step 1 to step N.

**If you want perfectly stable PRBs across the episode**, set `demand_calibration_alpha=0.0`  
(calibration disabled) — then useful_prbs will still vary slightly due to channel noise  
and queue transients, but the large monotonic drift will disappear.

### Q3 — After handover, do PRBs increase or decrease?

**It depends on the SINR differential between source and target cell:**

| UE moves to | SINR | bits/PRB | PRBs needed for same demand | Net effect |
|---|---|---|---|---|
| Better cell | ↑ | ↑ | ↓ | Total useful PRBs **decrease** transiently |
| Worse cell | ↓ | ↓ | ↑ | Total useful PRBs **increase** transiently |

The `_calibrate_demand_from_radio_window()` will correct this over the next 2–3 windows.  
Note that **`_recalibrate_handover_ues()` is defined but has NO caller** — it is dead code.  
This means the old `source.bit_rate` (calibrated for the source cell) is used for the first  
post-HO window, which amplifies the transient.

**Demand PRBs (`upper_demand_prbs`) are always conserved** — the total never changes, only the per-gNB distribution.